# 03 — Évaluer et expliquer le modèle de référence v2

Le 01 prépare neuf variables métier ; le 02 entraîne et archive les modèles. Ce notebook charge le dernier principal **v2 terminé et complet**, avec ses propres données, colonnes et seuil.

Aucun fit, aucune création de feature et aucun nouveau choix de seuil. Les onze figures couvrent calibration, distribution, seuil, erreurs, gains/lift, importances, SHAP, ROC/précision-rappel et bootstrap par client. Les données ont déjà été explorées : le test reste rétrospectif.

Les figures SVG et les tables sont conservées dans un run d'évaluation lié à l'entraînement. Les v1 restent dans MLflow mais ne sont pas chargées automatiquement par cette refonte.


## 1 — Imports et paramètres

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import EFstrType, Pool
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib.patches import Patch

# sklearn accepte zero_division=0 ; son annotation 1.9 le restreint à str.
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

if not Path("settings.py").is_file() and Path("../settings.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from settings import settings

SEED = 42
FIGSIZE = (11, 5)
SHAP_SAMPLE_SIZE = 500
SHAP_TOP_N = 15
LOCAL_ROW_POSITION = 0
import hashlib
import json
import os
import tempfile
from io import BytesIO

os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")
import mlflow
import mlflow.catboost as mlflow_catboost
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
if not (settings.MLFLOW_DIR / "mlflow.db").is_file():
    raise FileNotFoundError(
        "Aucune base MLflow locale ; exécuter le notebook 02 jusqu'à la sauvegarde MLflow."
    )
client = MlflowClient()
# Figures conservées en mémoire, puis logguées ensemble après évaluation réussie.
evaluation_figures = {}

from io import StringIO

from IPython.display import HTML

FIG_WIDTH = 11  # Largeur de référence en pouces, ajustable.
PANEL_HEIGHT = 3.8
plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.alpha": 0.2,
        "figure.dpi": 110,
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def show_figure(fig, name):
    """Affichage SVG qui s'adapte à la largeur disponible sans écrire de fichier."""
    fig.tight_layout()
    buffer = StringIO()
    fig.savefig(buffer, format="svg", bbox_inches="tight")
    svg = buffer.getvalue()
    evaluation_figures[name] = svg
    svg = svg.replace("<svg ", '<svg style="display:block;width:100%;height:auto;" ', 1)
    display(HTML('<div style="width:100%;max-width:100%;">' + svg + "</div>"))
    plt.close(fig)

## 2 — Charger le dernier principal v2 complet

Le filtre retient seulement `FINISHED`, `role=principal`, `contract_version=2`, `artifacts_complete=true`, par date de fin décroissante. Le run sélectionné est figé pour la session ; les diagnostics et anciens modèles v1 ne le remplacent pas. Les données archivées sont vérifiées par SHA-256.

In [ ]:
experiment = client.get_experiment_by_name(settings.MLFLOW_EXPERIMENT)
if experiment is None:
    raise FileNotFoundError(
        "Aucun entraînement MLflow : exécuter le notebook 02 jusqu'à sa sauvegarde MLflow."
    )
candidates = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED' AND tags.stage = 'training' AND tags.role = 'principal' AND tags.contract_version = '2' AND tags.artifacts_complete = 'true'",
    order_by=["attributes.end_time DESC", "attributes.start_time DESC"],
    max_results=1,
)
if not candidates:
    raise FileNotFoundError(
        "Aucun modèle principal v2 terminé et complet ; exécuter la sauvegarde MLflow du 02."
    )
training_run = candidates[0]
training_run_id = training_run.info.run_id
with tempfile.TemporaryDirectory(prefix="telco-contract-") as temporary:
    contract_path = client.download_artifacts(training_run_id, "contract.json", temporary)
    contract = json.loads(Path(contract_path).read_text())
if contract.get("contract_version") != 2:
    raise ValueError("Version de contrat MLflow non prise en charge.")
MODEL_FEATURES = contract["model_features"]
CATEGORICAL_FEATURES = contract["categorical_features"]
NEW_FEATURES = contract["new_features"]
EXCLUDED = contract["excluded"]
best_threshold = float(contract["threshold"])
if not 0 <= best_threshold <= 1:
    raise ValueError("Seuil archivé invalide.")
model = mlflow_catboost.load_model(contract["model_uri"])
if list(model.feature_names_) != MODEL_FEATURES:
    raise ValueError("Le schéma du modèle diffère du contrat archivé.")
if {MODEL_FEATURES[i] for i in model.get_cat_feature_indices()} != set(CATEGORICAL_FEATURES):
    raise ValueError("Les catégories du modèle diffèrent du contrat archivé.")
print("Run principal chargé :", training_run_id)
print("Modèle :", contract["model_uri"])
print("Seuil verrouillé :", best_threshold)
display(
    pd.DataFrame(
        {
            "variable": MODEL_FEATURES,
            "catégorielle": [c in CATEGORICAL_FEATURES for c in MODEL_FEATURES],
        }
    )
)

In [ ]:
def load_split(name: str) -> pd.DataFrame:
    dataset = contract["datasets"][name]
    with tempfile.TemporaryDirectory(prefix="telco-dataset-") as temporary:
        path = Path(client.download_artifacts(training_run_id, dataset["artifact_path"], temporary))
        data_bytes = path.read_bytes()
    if hashlib.sha256(data_bytes).hexdigest() != dataset["sha256"]:
        raise ValueError(f"{name} : les données archivées ne correspondent pas au SHA-256 du run.")
    frame = pd.read_csv(
        BytesIO(data_bytes),
        dtype={"customer_id": "string", "dept": "string"},
        parse_dates=["snapshot_date"],
    )
    if len(frame) != dataset["rows"]:
        raise ValueError(f"{name} : nombre de lignes différent de celui du run.")
    required = set(MODEL_FEATURES) | {"churn", "customer_id", "snapshot_date"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} : colonnes absentes {sorted(missing)}")
    if (
        frame.empty
        or frame.churn.isna().any()
        or not frame.churn.isin([0, 1]).all()
        or frame.churn.nunique() != 2
    ):
        raise ValueError(f"{name} : deux classes binaires non manquantes sont requises.")
    if frame.snapshot_date.isna().any() or frame.customer_id.isna().any():
        raise ValueError(f"{name} : date ou identifiant manquant.")
    if frame.duplicated(["customer_id", "snapshot_date"]).any():
        raise ValueError(f"{name} : clés client × snapshot dupliquées.")
    return frame


def make_matrix(part: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    x = part[features].copy()

    for column in features:
        if column in CATEGORICAL_FEATURES:
            x[column] = x[column].astype("string").fillna("__MISSING__").astype(str)
        else:
            x[column] = pd.to_numeric(x[column], errors="coerce").astype(float)
            x[column] = x[column].replace([np.inf, -np.inf], np.nan)

    return x


val_df = load_split("validation")
X_val = make_matrix(val_df, MODEL_FEATURES)
y_val = val_df.churn.to_numpy(dtype=int)
print("Validation archivée :", X_val.shape)

## 3 — Calculer les probabilités de validation avec le modèle chargé

Uniquement `predict_proba` : aucun appel à `fit`.

In [ ]:
# Prédiction uniquement : le modèle vient de MLflow, aucun fit dans ce notebook.
p_val = model.predict_proba(X_val)[:, 1]
print("Nombre d'arbres du modèle chargé :", model.tree_count_)

## 4 — Calibration descriptive et distribution des probabilités

Une probabilité bien calibrée de 0,30 correspond à environ 30 % de churn observé. Les classes de probabilité sont fixes ; seuls les groupes non vides sont affichés. Le tableau donne leurs effectifs et l'écart signé (observé − prédit). Aucun calibrateur n'est ajusté ici.

In [ ]:
calibration = pd.DataFrame({"probability": p_val, "churn": y_val})
calibration["bin"] = pd.cut(calibration.probability, np.linspace(0, 1, 11).tolist(), include_lowest=True)
calibration_table = (
    calibration.groupby("bin", observed=True)
    .agg(
        count=("churn", "size"),
        mean_probability=("probability", "mean"),
        observed_churn=("churn", "mean"),
    )
    .reset_index()
)
calibration_table["gap"] = calibration_table.observed_churn - calibration_table.mean_probability
calibration_table["absolute_gap"] = calibration_table.gap.abs()
display(calibration_table)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Calibration idéale")
axes[0].plot(
    calibration_table.mean_probability,
    calibration_table.observed_churn,
    "o-",
    color="#378ADD",
    label="CatBoost",
)
axes[0].set(
    title="Calibration — validation",
    xlabel="Probabilité moyenne prédite",
    ylabel="Fréquence de churn",
    xlim=(0, 1),
    ylim=(0, 1),
)
axes[0].legend()
axes[1].hist(p_val, bins=20, color="#378ADD", edgecolor="white", alpha=0.85)
axes[1].set(
    title="Distribution des probabilités — validation", xlabel="P(churn)", ylabel="Snapshots"
)
show_figure(fig, "calibration")

## 5 — Seuil : précision, rappel et volumes d'erreurs

Le seuil a été choisi dans le 02 et est chargé depuis MLflow. La grille ci-dessous sert uniquement à visualiser sa sensibilité ; elle ne le remplace pas. La ligne verticale indique ce seuil verrouillé pour la suite. Les volumes sont des **snapshots de validation**, pas des clients uniques ni des contacts mensuels. Ce seuil statistique n'est pas une politique économique top 300.

In [ ]:
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 181):
    pred = (p_val >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    threshold_rows.append(
        {
            "threshold": threshold,
            "MCC": matthews_corrcoef(y_val, pred),
            "precision": precision_score(y_val, pred, zero_division=0),  # pyright: ignore[reportArgumentType]
            "recall": recall_score(y_val, pred, zero_division=0),  # pyright: ignore[reportArgumentType]
            "TP": tp,
            "FN": fn,
            "FP": fp,
        }
    )
tradeoff = pd.DataFrame(threshold_rows)
print(f"Seuil chargé depuis le run principal : {best_threshold:.3f}")
display(tradeoff.loc[np.isclose(tradeoff.threshold, best_threshold)])
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
axes[0].plot(tradeoff.threshold, tradeoff.recall, color="#C44E52", label="Rappel")
axes[0].plot(tradeoff.threshold, tradeoff.precision, color="#4C72B0", label="Précision")
axes[0].set(
    title="Compromis précision / rappel — validation",
    xlabel="Seuil",
    ylabel="Score",
    ylim=(0, 1.02),
)
for column, color, label in [
    ("TP", "#C44E52", "Churns détectés (TP)"),
    ("FN", "#DD8452", "Churns manqués (FN)"),
    ("FP", "#4C72B0", "Fausses alertes (FP)"),
]:
    axes[1].plot(tradeoff.threshold, tradeoff[column], color=color, label=label)
axes[1].set(title="Volumes d'erreurs — validation", xlabel="Seuil", ylabel="Snapshots")
for ax in axes:
    ax.axvline(best_threshold, color="black", linestyle="--", label=f"Seuil = {best_threshold:.3f}")
    ax.legend()
show_figure(fig, "threshold_tradeoff")

## 6 — Gains cumulés et lift par décile

Tri par probabilité décroissante ; le décile 1 contient les risques les plus élevés. Les groupes ont des effectifs aussi proches que possible. Les égalités de score sont départagées par l'ordre des lignes, jamais par la cible. Le lift est le taux de churn du décile divisé par la prévalence globale de validation. Le gain cumulé est la part des churns capturés ; ce n'est pas un gain monétaire ni un effet d'une campagne.

In [ ]:
ranked = (
    pd.DataFrame({"probability": p_val, "churn": y_val})
    .sort_values("probability", ascending=False, kind="stable")
    .reset_index(drop=True)
)
ranked["decile"] = np.minimum(np.arange(len(ranked)) * 10 // len(ranked) + 1, 10)
lift_table = ranked.groupby("decile").agg(count=("churn", "size"), churns=("churn", "sum"))
lift_table["churn_rate"] = lift_table.churns / lift_table["count"]
lift_table["lift"] = lift_table.churn_rate / y_val.mean()
lift_table["cumulative_population"] = lift_table["count"].cumsum() / len(ranked)
lift_table["cumulative_churns"] = lift_table.churns.cumsum() / y_val.sum()
display(lift_table)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
axes[0].plot(
    np.r_[0, lift_table.cumulative_population],
    np.r_[0, lift_table.cumulative_churns],
    "o-",
    linewidth=2.5,
    label="CatBoost",
)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Aléatoire")
axes[0].set(
    title="Gains cumulés — validation",
    xlabel="Part des snapshots ciblés",
    ylabel="Part cumulée des churns détectés",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
axes[0].legend()
axes[1].bar(lift_table.index.astype(str), lift_table.lift, color="#378ADD")
axes[1].axhline(1, color="black", linestyle="--", alpha=0.5)
axes[1].set(title="Lift par décile (1 = risque le plus élevé)", xlabel="Décile", ylabel="Lift")
show_figure(fig, "lift_and_gains")

## 7 — Importance native

Les variables créées sont violettes, les sources sont orange. Les barres mesurent une importance relative normalisée, pas le sens de l'effet ni la part des churns expliquée. Les variables corrélées peuvent partager leur importance ; une faible valeur n'autorise pas seule une suppression.

In [ ]:
native_importance = pd.Series(
    model.get_feature_importance(type=EFstrType.PredictionValuesChange),
    index=MODEL_FEATURES,
    name="importance",
).sort_values(ascending=False)
display(native_importance.to_frame())
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(6, len(native_importance) * 0.3)))
colors = ["#6366F1" if f in NEW_FEATURES else "#F05B46" for f in native_importance.index]
bars = ax.barh(native_importance.index, native_importance.to_numpy(), color=colors)
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.invert_yaxis()
ax.legend(
    handles=[
        Patch(facecolor="#6366F1", label="Variable créée (feature engineering)"),
        Patch(facecolor="#F05B46", label="Variable d’origine du dataset"),
    ],
    title="Origine des variables",
    loc="lower right",
)
ax.margins(x=0.12)
ax.set(title="Importance native — CatBoost", xlabel="PredictionValuesChange")
show_figure(fig, "native_importance")

## 8 — SHAP natifs : calcul et vérification

Échantillon reproductible de 500 snapshots au maximum, tiré **sans utiliser la cible** dans la validation. CatBoost fournit les contributions TreeSHAP et la valeur de base. Pour chaque ligne : `score_brut = base + somme(SHAP)` et `P(churn) = sigmoid(score_brut)`. Les contributions sont en **log-odds**, pas en points de probabilité. Positif augmente le score de churn, négatif le diminue. Les variables corrélées peuvent se partager l'attribution ; aucune interprétation causale ne découle de ces vues.

In [ ]:
X_shap = X_val.sample(n=min(SHAP_SAMPLE_SIZE, len(X_val)), random_state=SEED).copy()
shap_pool = Pool(X_shap, cat_features=CATEGORICAL_FEATURES)
shap_array = np.asarray(model.get_feature_importance(shap_pool, type=EFstrType.ShapValues))
shap_values = pd.DataFrame(shap_array[:, :-1], columns=MODEL_FEATURES, index=X_shap.index)
shap_base = shap_array[:, -1]
raw_score = np.asarray(model.predict(X_shap, prediction_type="RawFormulaVal"))
np.testing.assert_allclose(
    shap_base + shap_values.sum(axis=1).to_numpy(), raw_score, rtol=1e-6, atol=1e-6
)
shap_importance = shap_values.abs().mean().sort_values(ascending=False)
top_features = shap_importance.head(SHAP_TOP_N).index.tolist()
print("Additivité SHAP vérifiée ; snapshots :", len(X_shap))

## 8.1 — Importance SHAP globale

Moyenne des valeurs absolues des contributions sur l’échantillon de validation.

In [ ]:
display(shap_importance.to_frame("mean_abs_SHAP"))
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.barh(top_features, shap_importance.loc[top_features].to_numpy(), color="#BA7517")
ax.invert_yaxis()
ax.set(title="Importance SHAP globale — CatBoost", xlabel="Moyenne de |SHAP| (log-odds)")
show_figure(fig, "shap_global")

## 8.2 — Résumé SHAP

Un point représente un snapshot. Les lignes sont triées par importance SHAP. Pour les variables numériques, la couleur est normalisée **séparément par variable** (bleu = faible, rouge = élevée) ; les valeurs manquantes sont grises. Les catégories sont également grises : un code arbitraire de catégorie n'a pas d'ordre numérique. La dispersion verticale sert uniquement à rendre les points visibles.

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(6, len(top_features) * 0.4)))
rng = np.random.default_rng(SEED)
for position, feature in enumerate(top_features):
    raw = X_shap[feature]
    y_jitter = position + rng.normal(0, 0.09, len(raw))
    values = shap_values[feature].to_numpy()
    if feature not in CATEGORICAL_FEATURES:
        numeric = pd.to_numeric(raw, errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(numeric)
        color_values = np.full(len(raw), 0.5)
        if valid.any() and np.ptp(numeric[valid]) > 0:
            color_values[valid] = (numeric[valid] - numeric[valid].min()) / np.ptp(numeric[valid])
        ax.scatter(
            values[valid],
            y_jitter[valid],
            c=color_values[valid],
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            s=18,
            alpha=0.65,
        )
        ax.scatter(values[~valid], y_jitter[~valid], color="grey", s=18, alpha=0.65)
    else:
        ax.scatter(values, y_jitter, color="grey", s=18, alpha=0.65)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(top_features)), top_features)
ax.invert_yaxis()
ax.set(title="Résumé SHAP — CatBoost", xlabel="Contribution au score de churn (log-odds)")
colorbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=Normalize(0, 1), cmap="coolwarm"), ax=ax, pad=0.01
)
colorbar.set_ticks([0, 1], labels=["Faible", "Élevée"])
colorbar.set_label("Valeur numérique relative à chaque variable")
show_figure(fig, "shap_summary")

## 8.3 — Explication locale

Choisir `LOCAL_ROW_POSITION` dans la configuration, puis réexécuter cette cellule. Les barres rouges augmentent le score ; les bleues le diminuent. La somme des variables non affichées est conservée dans « Autres variables », pour garder une explication complète. Il s'agit de contributions centrées sur zéro, comme dans la référence, et non d'un waterfall cumulatif.

In [ ]:
if not 0 <= LOCAL_ROW_POSITION < len(X_shap):
    raise ValueError("LOCAL_ROW_POSITION hors de l'échantillon SHAP.")
local_values = shap_values.iloc[LOCAL_ROW_POSITION]
local_features = X_shap.iloc[LOCAL_ROW_POSITION]
selected = local_values.abs().nlargest(SHAP_TOP_N).index
local_plot = local_values.loc[selected].copy()
labels = {feature: f"{feature} = {local_features[feature]}" for feature in selected}
if len(selected) < len(local_values):
    local_plot.loc["__other__"] = local_values.drop(selected).sum()
    labels["__other__"] = "Autres variables (somme)"
local_plot = local_plot.sort_values(ascending=False)
probability = model.predict_proba(X_shap.iloc[[LOCAL_ROW_POSITION]])[0, 1]
print(
    f"Base : {shap_base[LOCAL_ROW_POSITION]:.4f} ; score brut : {raw_score[LOCAL_ROW_POSITION]:.4f} ; P(churn) : {probability:.4f}"
)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(6, len(local_plot) * 0.4)))
ax.barh(
    [labels[f] for f in local_plot.index],
    local_plot.to_numpy(),
    color=np.where(local_plot.to_numpy() >= 0, "#E24B4A", "#378ADD"),
)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set(
    title=f"Explication locale SHAP — snapshot d'index {local_features.name}",
    xlabel="Contribution au score brut de churn (log-odds)",
)
show_figure(fig, "shap_local")

## 8.4 — Dépendance SHAP

Par défaut, la variable ayant la plus forte importance SHAP. Changer `DEPENDENCE_FEATURE` pour en examiner une autre. Pour les catégories, l'axe affiche les modalités sans supposer un ordre métier. Les valeurs numériques manquantes sont comptées et exclues du nuage uniquement.

In [ ]:
DEPENDENCE_FEATURE = top_features[0]
if DEPENDENCE_FEATURE not in MODEL_FEATURES:
    raise ValueError("Variable absente des entrées du modèle.")
raw = X_shap[DEPENDENCE_FEATURE]
fig, ax = plt.subplots(figsize=FIGSIZE)
if DEPENDENCE_FEATURE in CATEGORICAL_FEATURES:
    categories = pd.Categorical(raw)
    x_values = categories.codes
    ax.set_xticks(range(len(categories.categories)), categories.categories, rotation=30)
else:
    x_values = pd.to_numeric(raw, errors="coerce").to_numpy(dtype=float)
    print("Valeurs manquantes non affichées :", int(np.isnan(x_values).sum()))
ax.scatter(x_values, shap_values[DEPENDENCE_FEATURE], alpha=0.55, color="#378ADD")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(
    title=f"Dépendance SHAP — {DEPENDENCE_FEATURE} — CatBoost",
    xlabel=DEPENDENCE_FEATURE,
    ylabel="Contribution au score de churn (log-odds)",
)
show_figure(fig, "shap_dependence")

## 9 — Évaluation sur le test temporel

Exécuter après les diagnostics de validation. Le modèle et le seuil sont conservés tels quels. Aucun réglage après consultation de ces scores : une confirmation aveugle demanderait une nouvelle période. Les libellés sont adaptés au churn, sans reprendre les classes de défaut de crédit du projet de référence.

In [ ]:
test_df = load_split("test")
if val_df.snapshot_date.max() >= test_df.snapshot_date.min():
    raise ValueError("Les périodes validation et test se chevauchent.")
X_test = make_matrix(test_df, MODEL_FEATURES)
y_test = test_df.churn.to_numpy(dtype=int)
p_test = model.predict_proba(X_test)[:, 1]
pred_test = (p_test >= best_threshold).astype(int)
metrics = pd.Series(
    {
        "Average Precision": average_precision_score(y_test, p_test),
        "ROC-AUC": roc_auc_score(y_test, p_test),
        "MCC": matthews_corrcoef(y_test, pred_test),
        "Précision": precision_score(y_test, pred_test, zero_division=0),  # pyright: ignore[reportArgumentType]
        "Rappel": recall_score(y_test, pred_test, zero_division=0),  # pyright: ignore[reportArgumentType]
        "F1": f1_score(y_test, pred_test, zero_division=0),  # pyright: ignore[reportArgumentType]
        "Logloss": log_loss(y_test, p_test, labels=[0, 1]),
        "Brier": brier_score_loss(y_test, p_test),
    },
    name="Test",
)
display(metrics.to_frame())
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_test, labels=[0, 1]), display_labels=["Sans churn", "Churn"]
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Matrice de confusion — test — seuil={best_threshold:.3f}")
axes[0].set(xlabel="Classe prédite", ylabel="Classe réelle")
fpr, tpr, _ = roc_curve(y_test, p_test)
axes[1].plot(fpr, tpr, linewidth=2.5, label=f"CatBoost (AUC={metrics['ROC-AUC']:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Aléatoire")
axes[1].set(
    title="Courbe ROC — test", xlabel="Taux de faux positifs", ylabel="Taux de vrais positifs"
)
axes[1].legend(loc="lower right")
show_figure(fig, "test_confusion_and_roc")

### 9.1 — Courbes précision-rappel

Les courbes de validation et test complètent le ROC lorsque les churners sont rares. La référence horizontale est la prévalence du split. Elles ne servent à aucun nouveau réglage du modèle ou du seuil.

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
for label, target, probability, color in [
    ("Validation", y_val, p_val, "#2563EB"),
    ("Test", y_test, p_test, "#E11D48"),
]:
    precision, recall, _ = precision_recall_curve(target, probability)
    ax.plot(
        recall,
        precision,
        color=color,
        label=f"{label} (AP={average_precision_score(target, probability):.3f})",
    )
    ax.axhline(target.mean(), color=color, linestyle="--", alpha=0.5)
ax.set(title="Courbes précision-rappel", xlabel="Rappel", ylabel="Précision")
ax.legend()
show_figure(fig, "precision_recall")

### 9.2 — Incertitude du test : bootstrap par client

Analyse déplacée depuis le notebook de modélisation. On rééchantillonne les clients et tous leurs snapshots, sans réentraîner. Les intervalles percentile à 95 % sont descriptifs ; ils n'incluent pas l'incertitude du réentraînement ou de la dérive.

In [ ]:
# Bootstrap clusterisé, reproductible et entièrement en mémoire.
test_predictions = test_df[["customer_id", "snapshot_date", "churn"]].copy().reset_index(drop=True)
test_predictions["probability"] = p_test
client_positions = test_predictions.groupby("customer_id", sort=False).indices
client_ids = np.array(list(client_positions), dtype=object)
rng = np.random.default_rng(SEED)
B = 400
bootstrap_ap, bootstrap_brier = [], []
for _ in range(B):
    sampled_clients = rng.choice(client_ids, size=len(client_ids), replace=True)
    positions = np.concatenate([client_positions[cid] for cid in sampled_clients])
    sampled = test_predictions.iloc[positions]
    if sampled["churn"].nunique() < 2:
        continue
    bootstrap_ap.append(average_precision_score(sampled["churn"], sampled["probability"]))
    bootstrap_brier.append(brier_score_loss(sampled["churn"], sampled["probability"]))

ci = pd.DataFrame(
    [
        {
            "Métrique": "Average Precision",
            "Estimation": average_precision_score(y_test, p_test),
            "IC 95 % bas": np.quantile(bootstrap_ap, 0.025),
            "IC 95 % haut": np.quantile(bootstrap_ap, 0.975),
        },
        {
            "Métrique": "Brier",
            "Estimation": brier_score_loss(y_test, p_test),
            "IC 95 % bas": np.quantile(bootstrap_brier, 0.025),
            "IC 95 % haut": np.quantile(bootstrap_brier, 0.975),
        },
    ]
)
display(ci)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
for ax, values, color, title in [
    (axes[0], bootstrap_ap, "#2563EB", "Average Precision"),
    (axes[1], bootstrap_brier, "#059669", "Brier"),
]:
    ax.hist(values, bins=25, color=color, alpha=0.8)
    ax.set(title=f"Bootstrap clients — {title}", xlabel=title, ylabel="Réplications")
show_figure(fig, "test_bootstrap")

## 9 bis — Logguer l'évaluation dans MLflow

Cette cellule crée un run d'évaluation lié au run d'entraînement affiché au début. Elle conserve les scores, le seuil, les tables de diagnostic, les prédictions du test et les onze graphiques. Elle ne sauvegarde pas un nouveau modèle et ne modifie pas le run d'entraînement. Une réexécution crée une nouvelle évaluation. `data/output/` reste réservé au futur fichier de scoring final.


In [ ]:
if mlflow.active_run() is not None:
    raise RuntimeError("Terminer le run MLflow actif avant de logguer l'évaluation.")
with mlflow.start_run(
    experiment_id=experiment.experiment_id,
    run_name="evaluation_catboost",
    tags={"stage": "evaluation", "role": "evaluation", "training_run_id": training_run_id},
) as evaluation_run:
    mlflow.log_params(
        {
            "training_run_id": training_run_id,
            "decision_threshold": best_threshold,
            "feature_count": len(MODEL_FEATURES),
            "shap_sample_rows": len(X_shap),
        }
    )
    mlflow.log_metrics(
        {
            "test_average_precision": float(metrics["Average Precision"]),
            "test_roc_auc": float(metrics["ROC-AUC"]),
            "test_mcc": float(metrics["MCC"]),
            "test_precision": float(metrics["Précision"]),
            "test_recall": float(metrics["Rappel"]),
            "test_f1": float(metrics["F1"]),
            "test_log_loss": float(metrics["Logloss"]),
            "test_brier": float(metrics["Brier"]),
        }
    )
    mlflow.log_dict(
        {
            "training_run_id": training_run_id,
            "model_uri": contract["model_uri"],
            "threshold": best_threshold,
            "datasets": contract["datasets"],
        },
        "evaluation_reference.json",
    )
    tables = {
        "test_bootstrap": ci,
        "calibration": calibration_table,
        "threshold_tradeoff": tradeoff,
        "lift_and_gains": lift_table,
        "native_importance": native_importance.to_frame(),
        "shap_importance": shap_importance.to_frame("mean_abs_SHAP"),
    }
    for name, table in tables.items():
        mlflow.log_text(table.to_csv(index=True), f"tables/{name}.csv")
    test_predictions = test_df[["customer_id", "snapshot_date", "churn"]].copy()
    test_predictions["probability"] = p_test
    test_predictions["prediction"] = pred_test
    mlflow.log_text(test_predictions.to_csv(index=False), "predictions/test.csv")
    for name, svg in evaluation_figures.items():
        mlflow.log_text(svg, f"figures/{name}.svg")
    snapshot = json.loads(
        (
            settings.PROJECT_ROOT / "notebooks" / "03_churn_evaluation_et_explicabilite.ipynb"
        ).read_text()
    )
    for cell in snapshot["cells"]:
        if cell["cell_type"] == "code":
            cell.update(outputs=[], execution_count=None)
    mlflow.log_dict(snapshot, "source/notebook_03.ipynb")
    mlflow.set_tag("contract_version", "2")
    evaluation_run_id = evaluation_run.info.run_id
print("Évaluation MLflow sauvegardée :", evaluation_run_id)

## 10 — Observations à compléter

- Les probabilités sont-elles fiables ou seulement discriminantes ?
- Combien de snapshots déclenchent une alerte au seuil retenu ?
- La facture et ses dérivées dominent-elles les SHAP ? Comparer une variante sans cette famille sur validation, sans ajuster sur le test.
- Les explications sont-elles stables entre profils et périodes ?
- Le lift cumulé ne démontre ni l'efficacité d'une offre ni la rentabilité d'une campagne.

Le ciblage mensuel à 300 contacts, le gain économique et le fichier de prédictions final restent des étapes distinctes. `data/output/` reste réservé à ce dernier fichier.